In [ ]:
import mc_postgres_db.models as models
from dotenv import load_dotenv
import os
import polars as pl
import numpy as np
import datetime as dt
from sqlalchemy import create_engine
from sqlalchemy import select, alias
import sys
from pathlib import Path
import importlib
from tqdm.notebook import tqdm
import plotly.graph_objects as go
from dask.distributed import Client, LocalCluster, as_completed
from stats import rolling_pairs_trading

# Add the workspace root to Python path so we can import from src
workspace_root = (
    Path(__file__).parent.parent.parent
    if "__file__" in globals()
    else Path.cwd().parent.parent
)
sys.path.insert(0, str(workspace_root))

import src.utils.stochastic as stochastic

# After making changes to your_module.py
importlib.reload(stochastic)


load_dotenv()

force_recompute = True
engine = create_engine(os.getenv("POSTGRES_URL"))

In [ ]:
# Define trading parameters for OU model
STOP_LOSS_FACTOR = 2.25
DISCOUNT_RATE = 0.0001  # Discount rate
TRANSACTION_COST = 0.001  # Transaction cost
P_VALUE_THRESHOLD = 0.01  # Only trade if p_value < 0.01 (99% confidence)

In [ ]:
pairs_trading_stats_df = pl.read_parquet(
    "../pairs-trading/cointegrated_pairs_trading_stats.parquet"
).filter(pl.col("linear_fit_beta") > 0)
pairs_trading_stats_df

In [ ]:
# Set up Dask cluster
cluster = LocalCluster(n_workers=8, threads_per_worker=1)
client = Client(cluster)

In [ ]:
with engine.connect() as conn:
    from_asset_id = conn.execute(
        select(models.Asset.id).where(models.Asset.name == "USD")
    ).scalar_one_or_none()
    if from_asset_id is None:
        raise ValueError("USD asset not found")

In [ ]:
with engine.connect() as conn:
    ignore_asset_ids = pl.read_database(
        select(models.Asset).where(models.Asset.underlying_asset_id == from_asset_id),
        engine,
    )["id"].to_list()
    display(ignore_asset_ids)

In [ ]:
from_asset = alias(models.Asset, "from_asset")
to_asset = alias(models.Asset, "to_asset")
provider_asset_group_df = pl.read_database(
    select(
        models.ProviderAssetGroupMember.provider_asset_group_id,
        models.ProviderAssetGroupMember.provider_id,
        models.ProviderAssetGroupMember.from_asset_id,
        from_asset.c.name.label("from_asset_name"),
        models.ProviderAssetGroupMember.to_asset_id,
        to_asset.c.name.label("to_asset_name"),
    )
    .join(
        models.ProviderAssetGroup,
        models.ProviderAssetGroupMember.provider_asset_group_id
        == models.ProviderAssetGroup.id,
    )
    .join(
        from_asset,
        models.ProviderAssetGroupMember.from_asset_id == from_asset.c.id,
    )
    .join(
        to_asset,
        models.ProviderAssetGroupMember.to_asset_id == to_asset.c.id,
    )
    .where(
        models.ProviderAssetGroup.is_active == True,
        to_asset.c.id.notin_(ignore_asset_ids),
        from_asset.c.id == from_asset_id,
    ),
    engine,
)
provider_asset_group_sample_ids = (
    provider_asset_group_df["provider_asset_group_id"]
    .unique()
    .sample(10, shuffle=True)
    .to_list()
)
provider_asset_group_df = provider_asset_group_df.filter(
    pl.col("provider_asset_group_id").is_in(provider_asset_group_sample_ids)
)
to_asset_ids = provider_asset_group_df["to_asset_id"].unique().sort().to_list()
from_asset_ids = provider_asset_group_df["from_asset_id"].unique().sort().to_list()
print(f"There are {len(to_asset_ids)} to assets")
print(f"There are {len(from_asset_ids)} from assets")
provider_asset_group_df

In [ ]:
window_days = 7
window = window_days * 24 * 60
test_days = 1
lookback_days = window_days + test_days
end_time = dt.datetime.combine(dt.datetime.today(), dt.time.min) - dt.timedelta(days=1)
start_time = end_time - dt.timedelta(days=lookback_days)
print(f"Start time: {start_time}")
print(f"End time: {end_time}")
from_asset = alias(models.Asset, "from_asset")
to_asset = alias(models.Asset, "to_asset")
provider = alias(models.Provider, "provider")
market_cache_path = Path("market_df.parquet")
if market_cache_path.exists() and not force_recompute:
    market_df = pl.read_parquet(market_cache_path)
else:
    market_df = pl.read_database(
        select(
            models.ProviderAssetMarket.timestamp,
            models.ProviderAssetMarket.provider_id,
            models.Provider.name.label("provider_name"),
            models.ProviderAssetMarket.from_asset_id,
            from_asset.c.name.label("from_asset_name"),
            models.ProviderAssetMarket.to_asset_id,
            to_asset.c.name.label("to_asset_name"),
            models.ProviderAssetMarket.close,
        )
        .join(
            models.Provider,
            models.ProviderAssetMarket.provider_id == models.Provider.id,
        )
        .join(from_asset, models.ProviderAssetMarket.from_asset_id == from_asset.c.id)
        .join(to_asset, models.ProviderAssetMarket.to_asset_id == to_asset.c.id)
        .where(
            models.ProviderAssetMarket.timestamp >= start_time,
            models.ProviderAssetMarket.timestamp <= end_time,
            models.ProviderAssetMarket.from_asset_id.in_(from_asset_ids),
            models.ProviderAssetMarket.to_asset_id.in_(to_asset_ids),
        ),
        engine,
    )
    market_df.write_parquet(market_cache_path)

In [ ]:
agg_timestamps = market_df.group_by("to_asset_id").agg(
    pl.col("timestamp").min().alias("min_timestamp"),
    pl.col("timestamp").max().alias("max_timestamp"),
)
start_timestamp = agg_timestamps["min_timestamp"].max()
end_timestamp = agg_timestamps["max_timestamp"].min()
print(f"Start timestamp: {start_timestamp}")
print(f"End timestamp: {end_timestamp}")
market_df = market_df.filter(
    pl.col("timestamp").is_between(start_timestamp, end_timestamp)
)
market_df

In [ ]:
provider_asset_group_df.join(market_df, on="to_asset_id", how="left")

In [ ]:
timestamps = market_df["timestamp"].sort().unique().to_list()
close_1 = (
    market_df.filter(
        pl.col("from_asset_id") == from_asset_ids[0],
        pl.col("to_asset_id") == to_asset_ids[0],
    )
    .select("timestamp", "close")
    .sort("timestamp")
)
close_2 = (
    market_df.filter(
        pl.col("from_asset_id") == from_asset_ids[0],
        pl.col("to_asset_id") == to_asset_ids[1],
    )
    .select("timestamp", "close")
    .sort("timestamp")
)
df_original = pl.DataFrame(
    {
        "timestamp": timestamps,
        "close_1": close_1["close"],
        "close_2": close_2["close"],
    }
)

In [ ]:
result = rolling_pairs_trading(
    df_original["close_1"].to_numpy(),
    df_original["close_2"].to_numpy(),
    window=window,
    p_value_threshold=P_VALUE_THRESHOLD,
)

In [ ]:
df = df_original.clone()

df = df.with_columns(
    pl.Series(
        name="p_value",
        values=result.p_value,
    ),
    pl.Series(
        name="alpha",
        values=result.alpha,
    ),
    pl.Series(
        name="beta",
        values=result.beta,
    ),
    pl.Series(
        name="residual_mean",
        values=result.residual_mean,
    ),
    pl.Series(
        name="residual_std",
        values=result.residual_std,
    ),
    pl.Series(
        name="ou_mu",
        values=result.ou_mu,
    ),
    pl.Series(
        name="ou_theta",
        values=result.ou_theta,
    ),
    pl.Series(
        name="ou_sigma",
        values=result.ou_sigma,
    ),
    pl.Series(
        name="half_life",
        values=result.half_life,
    ),
    pl.Series(
        name="L",
        values=result.residual_mean - STOP_LOSS_FACTOR * result.residual_std,
    ),
)
df

In [ ]:
df = df.drop_nans(["p_value"])
df = df.drop_nulls(["p_value"])
df

In [ ]:
# Filter to start of day (23:59)
df_start_of_day = df.filter(
    (pl.col("timestamp").dt.hour() == 23) & (pl.col("timestamp").dt.minute() == 59)
)

print(f"Total rows: {len(df)}")
print(f"Start of day rows (23:59): {len(df_start_of_day)}")


def compute_optimal_levels(
    p_value: float, ou_mu: float, ou_theta: float, ou_sigma: float, L: float
) -> tuple[float, float]:
    """Compute d_star (entry) and b_star (exit) from pre-computed OU parameters."""

    # Skip if not cointegrated or invalid OU params
    if p_value > P_VALUE_THRESHOLD or np.isnan(ou_mu) or np.isnan(ou_sigma):
        return (np.nan, np.nan)

    try:
        d_star = stochastic.OrnsteinUhlenbeck.get_optimal_entry_level(
            mu=ou_mu,
            sigma=ou_sigma,
            theta=ou_theta,
            r=DISCOUNT_RATE,
            c=TRANSACTION_COST,
            L=L,
        )
        b_star = stochastic.OrnsteinUhlenbeck.get_optimal_exit_level(
            mu=ou_mu,
            sigma=ou_sigma,
            theta=ou_theta,
            r=DISCOUNT_RATE,
            c=TRANSACTION_COST,
            L=L,
        )
        return (d_star, b_star)
    except Exception:
        return (np.nan, np.nan)


# Submit tasks to Dask (compute once, flip for direction 2 due to symmetry)
futures = [
    client.submit(compute_optimal_levels, pv, mu, theta, sigma, L)
    for pv, mu, theta, sigma, L in zip(
        df_start_of_day["p_value"].to_list(),
        df_start_of_day["ou_mu"].to_list(),
        df_start_of_day["ou_theta"].to_list(),
        df_start_of_day["ou_sigma"].to_list(),
        df_start_of_day["L"].to_list(),
    )
]

# Collect results preserving order
results = [None] * len(futures)
future_to_idx = {f: i for i, f in enumerate(futures)}
for future in tqdm(as_completed(futures), total=len(futures), desc="Optimal Levels"):
    results[future_to_idx[future]] = future.result()

client.close()
cluster.close()

# Extract d_star and b_star
d_star_values = [r[0] for r in results]
b_star_values = [r[1] for r in results]

# Add optimal level columns (symmetric - flip for direction 2)
# Direction 1: d_star is entry (below), b_star is exit (above)
# Direction 2: Flip the levels - b_star becomes entry, d_star becomes exit
df_final = df_start_of_day.with_columns(
    [
        pl.Series("d_star", d_star_values),
        pl.Series("b_star", b_star_values),
    ]
)

print(
    f"Rows with valid optimal levels: {df_final.filter(pl.col('d_star').is_not_nan()).height}"
)

In [ ]:
# Join with parameters from df_final
df_backtest = df_original.join(
    df_final[
        [
            "timestamp",
            "alpha",
            "beta",
            "p_value",
            "ou_mu",
            "ou_theta",
            "ou_sigma",
            "L",
            "d_star",
            "b_star",
        ]
    ],
    on="timestamp",
    how="left",
)

# Forward fill all parameters
df_backtest = df_backtest.with_columns(
    pl.col("alpha").forward_fill(),
    pl.col("beta").forward_fill(),
    pl.col("p_value").forward_fill(),
    pl.col("ou_mu").forward_fill(),
    pl.col("ou_theta").forward_fill(),
    pl.col("ou_sigma").forward_fill(),
    pl.col("L").forward_fill(),
    pl.col("d_star").forward_fill(),
    pl.col("b_star").forward_fill(),
)

# Calculate spread (single spread - directions determined by entry/exit level comparison)
df_backtest = df_backtest.with_columns(
    (pl.col("close_1") - pl.col("alpha") - pl.col("beta") * pl.col("close_2")).alias(
        "spread"
    ),
)

# Set spread to null when p_value > threshold
df_backtest = df_backtest.with_columns(
    pl.when(pl.col("p_value") > P_VALUE_THRESHOLD)
    .then(np.nan)
    .otherwise(pl.col("spread"))
    .alias("spread"),
)

# Rename columns for clarity
df_backtest = df_backtest.rename(
    {
        "L": "loss_level",
        "d_star": "entry_level",
        "b_star": "exit_level",
    }
)

# Drop nulls
df_backtest = df_backtest.drop_nulls(["alpha", "beta"])
df_backtest

In [ ]:
# To mirror the _1 levels about theta (mean), use theta - (x - theta) = 2*theta - x.
# This centers the flip around theta, not 0.
df_backtest = df_backtest.rename(
    {
        "loss_level": "loss_level_1",
        "entry_level": "entry_level_1",
        "exit_level": "exit_level_1",
    }
)
df_backtest = df_backtest.with_columns(
    (pl.col("ou_theta") - (pl.col("loss_level_1") - pl.col("ou_theta"))).alias(
        "loss_level_2"
    ),
    (pl.col("ou_theta") - (pl.col("entry_level_1") - pl.col("ou_theta"))).alias(
        "entry_level_2"
    ),
    (pl.col("ou_theta") - (pl.col("exit_level_1") - pl.col("ou_theta"))).alias(
        "exit_level_2"
    ),
)
df_backtest

In [ ]:
# Backtest the pairs trading strategy using df_backtest
# Strategy: Greedily enter whichever direction signals first, one trade at a time
#
# Single spread: spread = close_1 - alpha - beta * close_2
# Single p_value for cointegration validity
#
# Direction 1: spread goes DOWN → short close_2, long close_1
#   - Entry: spread crosses below entry_level_1
#   - Exit: spread crosses above exit_level_1 (take profit) or below loss_level_1 (stop loss)
#
# Direction 2: spread goes UP → short close_1, long close_2
#   - Entry: spread crosses above entry_level_2
#   - Exit: spread crosses below exit_level_2 (take profit) or above loss_level_2 (stop loss)
#
# IMPORTANT: Entry requires valid p_value (p_value <= P_VALUE_THRESHOLD)
#            Exit can occur even if p_value is invalid (uses forward-filled spread)

print("Configuration: Greedy Bidirectional Trading")
print("  - Will enter whichever direction signals first")
print("  - Only one position at a time")
print("  - Direction 1 (spread down): short close_2, long close_1")
print("  - Direction 2 (spread up): short close_1, long close_2")

# Check that df_backtest has required columns
required_cols = [
    "timestamp",
    "spread",
    "p_value",
    "entry_level_1",
    "exit_level_1",
    "loss_level_1",
    "entry_level_2",
    "exit_level_2",
    "loss_level_2",
    "close_1",
    "close_2",
]
missing_cols = [col for col in required_cols if col not in df_backtest.columns]
if missing_cols:
    print(f"ERROR: df_backtest is missing columns: {missing_cols}")
    print(f"Available columns: {df_backtest.columns}")
else:
    print(f"df_backtest has {len(df_backtest)} rows")

    # Initialize tracking variables
    initial_cash = 100000
    cash = initial_cash
    in_position = False
    current_direction = None  # Track which direction we're currently in (1 or 2)
    shares_short = 0
    shares_long = 0
    entry_price_short = 0
    entry_price_long = 0

    # Track previous spread to detect crosses
    last_spread = None

    # Track trades for analysis
    trade_history = []

    # Track skipped rows due to p_value
    skipped_rows = 0

    # Helper function to check if values are valid
    def is_valid(val):
        return val is not None and not np.isnan(val)

    # Iterate through each row in df_backtest
    for i in range(len(df_backtest)):
        row = df_backtest.slice(i, 1)

        # Get data (single spread and p_value, separate levels per direction)
        p_value = row["p_value"].item()
        spread = row["spread"].item()
        entry_level_1 = row["entry_level_1"].item()
        entry_level_2 = row["entry_level_2"].item()
        exit_level_1 = row["exit_level_1"].item()
        exit_level_2 = row["exit_level_2"].item()
        loss_level_1 = row["loss_level_1"].item()
        loss_level_2 = row["loss_level_2"].item()
        price_1 = row["close_1"].item()
        price_2 = row["close_2"].item()
        timestamp = row["timestamp"].item()

        # Check validity (single p_value, separate levels per direction)
        p_valid = is_valid(p_value) and p_value <= P_VALUE_THRESHOLD
        dir1_levels_valid = all(
            is_valid(v) for v in [spread, entry_level_1, exit_level_1, loss_level_1]
        )
        dir2_levels_valid = all(
            is_valid(v) for v in [spread, entry_level_2, exit_level_2, loss_level_2]
        )

        if not in_position:
            # Not in position - look for entry signal from either direction
            # Check both directions and take whichever signals first (priority to direction 1)

            entered = False

            # Direction 1: spread crosses DOWN below entry_level_1
            if not entered and p_valid and dir1_levels_valid:
                # Ensure entry_level is above loss_level
                loss_level_1_adj = loss_level_1
                if entry_level_1 <= loss_level_1:
                    loss_level_1_adj = entry_level_1 - 0.0001

                # Check for entry signal: spread crosses below entry_level
                if (
                    last_spread is not None
                    and last_spread >= entry_level_1
                    and spread < entry_level_1
                ):
                    # Entry signal for direction 1!
                    # Direction 1: short close_2, long close_1
                    price_short, price_long = price_2, price_1

                    position_value = initial_cash * 0.5
                    shares_short = position_value / price_short
                    shares_long = position_value / price_long

                    # Apply transaction costs
                    cash -= (shares_short * price_short * TRANSACTION_COST) + (
                        shares_long * price_long * TRANSACTION_COST
                    )

                    # Record entry
                    entry_price_short = price_short
                    entry_price_long = price_long
                    in_position = True
                    current_direction = 1
                    entered = True

                    trade_history.append(
                        {
                            "type": "entry",
                            "direction": 1,
                            "index": i,
                            "timestamp": timestamp,
                            "spread": spread,
                            "price_short": price_short,
                            "price_long": price_long,
                            "entry_level": entry_level_1,
                            "exit_level": exit_level_1,
                            "loss_level": loss_level_1,
                            "shares_short": shares_short,
                            "shares_long": shares_long,
                            "p_value": p_value,
                        }
                    )

            # Direction 2: spread crosses UP above entry_level_2
            if not entered and p_valid and dir2_levels_valid:
                # Ensure entry_level is below loss_level for direction 2 (spread going up)
                loss_level_2_adj = loss_level_2
                if entry_level_2 >= loss_level_2:
                    loss_level_2_adj = entry_level_2 + 0.0001

                # Check for entry signal: spread crosses above entry_level
                if (
                    last_spread is not None
                    and last_spread <= entry_level_2
                    and spread > entry_level_2
                ):
                    # Entry signal for direction 2!
                    # Direction 2: short close_1, long close_2
                    price_short, price_long = price_1, price_2

                    position_value = initial_cash * 0.5
                    shares_short = position_value / price_short
                    shares_long = position_value / price_long

                    # Apply transaction costs
                    cash -= (shares_short * price_short * TRANSACTION_COST) + (
                        shares_long * price_long * TRANSACTION_COST
                    )

                    # Record entry
                    entry_price_short = price_short
                    entry_price_long = price_long
                    in_position = True
                    current_direction = 2
                    entered = True

                    trade_history.append(
                        {
                            "type": "entry",
                            "direction": 2,
                            "index": i,
                            "timestamp": timestamp,
                            "spread": spread,
                            "price_short": price_short,
                            "price_long": price_long,
                            "entry_level": entry_level_2,
                            "exit_level": exit_level_2,
                            "loss_level": loss_level_2,
                            "shares_short": shares_short,
                            "shares_long": shares_long,
                            "p_value": p_value,
                        }
                    )

            if not entered and not p_valid:
                skipped_rows += 1

        else:
            # In position - look for exit signal based on current_direction
            if current_direction == 1:
                exit_level = exit_level_1
                loss_level = loss_level_1
                price_short, price_long = (
                    price_2,
                    price_1,
                )  # Direction 1: short close_2, long close_1
                levels_valid = dir1_levels_valid
            else:
                exit_level = exit_level_2
                loss_level = loss_level_2
                price_short, price_long = (
                    price_1,
                    price_2,
                )  # Direction 2: short close_1, long close_2
                levels_valid = dir2_levels_valid

            # Need valid spread and levels to check exit
            if not is_valid(spread) or not levels_valid:
                # Update last spread and continue
                if is_valid(spread):
                    last_spread = spread
                continue

            exit_signal = False
            exit_reason = None

            if current_direction == 1:
                # Direction 1: Exit on take profit (spread crosses UP above exit_level)
                if (
                    last_spread is not None
                    and last_spread <= exit_level
                    and spread > exit_level
                ):
                    exit_signal = True
                    exit_reason = "take_profit"
                # Direction 1: Exit on stop loss (spread crosses DOWN below loss_level)
                elif (
                    last_spread is not None
                    and last_spread >= loss_level
                    and spread < loss_level
                ):
                    exit_signal = True
                    exit_reason = "stop_loss"
            else:
                # Direction 2: Exit on take profit (spread crosses DOWN below exit_level)
                if (
                    last_spread is not None
                    and last_spread >= exit_level
                    and spread < exit_level
                ):
                    exit_signal = True
                    exit_reason = "take_profit"
                # Direction 2: Exit on stop loss (spread crosses UP above loss_level)
                elif (
                    last_spread is not None
                    and last_spread <= loss_level
                    and spread > loss_level
                ):
                    exit_signal = True
                    exit_reason = "stop_loss"

            if exit_signal:
                # Calculate P&L
                pnl_short = (entry_price_short - price_short) * shares_short
                pnl_long = (price_long - entry_price_long) * shares_long
                total_pnl = pnl_short + pnl_long

                # Apply transaction costs for exit
                cash += total_pnl
                cash -= (shares_short * price_short * TRANSACTION_COST) + (
                    shares_long * price_long * TRANSACTION_COST
                )

                trade_history.append(
                    {
                        "type": "exit",
                        "direction": current_direction,
                        "index": i,
                        "timestamp": timestamp,
                        "spread": spread,
                        "price_short": price_short,
                        "price_long": price_long,
                        "exit_level": exit_level,
                        "loss_level": loss_level,
                        "exit_reason": exit_reason,
                        "pnl_short": pnl_short,
                        "pnl_long": pnl_long,
                        "total_pnl": total_pnl,
                        "p_value": p_value,
                    }
                )

                # Reset position
                in_position = False
                current_direction = None
                shares_short = 0
                shares_long = 0
                entry_price_short = 0
                entry_price_long = 0

        # Update last spread for next iteration
        if is_valid(spread):
            last_spread = spread

    # Calculate final portfolio value
    final_portfolio_value = cash
    if in_position:
        # Mark to market if still in position
        if current_direction == 1:
            price_short, price_long = (
                price_2,
                price_1,
            )  # Direction 1: short close_2, long close_1
        else:
            price_short, price_long = (
                price_1,
                price_2,
            )  # Direction 2: short close_1, long close_2
        pnl_short = (entry_price_short - price_short) * shares_short
        pnl_long = (price_long - entry_price_long) * shares_long
        final_portfolio_value += pnl_short + pnl_long
        print(
            f"\nStill in position (Direction {current_direction}) at end. Unrealized P&L: {pnl_short + pnl_long:.2f}"
        )

    # Calculate returns
    total_return = final_portfolio_value - initial_cash
    return_pct = (total_return / initial_cash) * 100

    # Count trades by direction
    entries = [t for t in trade_history if t["type"] == "entry"]
    entries_dir1 = len([t for t in entries if t["direction"] == 1])
    entries_dir2 = len([t for t in entries if t["direction"] == 2])

    exits = [t for t in trade_history if t["type"] == "exit"]
    take_profits = len([t for t in exits if t["exit_reason"] == "take_profit"])
    stop_losses = len([t for t in exits if t["exit_reason"] == "stop_loss"])

    print("\n===== Backtest Results (Greedy Bidirectional) =====")
    print("\nPerformance:")
    print(f"  Initial Cash: ${initial_cash:,.2f}")
    print(f"  Final Portfolio Value: ${final_portfolio_value:,.2f}")
    print(f"  Total Return: ${total_return:,.2f} ({return_pct:.2f}%)")
    print("\nTrading Activity:")
    print(f"  Total Trades: {len(entries)}")
    print(
        f"    - Direction 1 (spread down, short close_2, long close_1): {entries_dir1}"
    )
    print(f"    - Direction 2 (spread up, short close_1, long close_2): {entries_dir2}")
    print(f"  Exits: {len(exits)}")
    print(f"    - Take Profits: {take_profits}")
    print(f"    - Stop Losses: {stop_losses}")
    print(f"  Rows Skipped (no valid p_value): {skipped_rows}")
    print(
        "\nNote: Exits can occur even when p_value is invalid (using forward-filled spread)"
    )

    # Convert trade_history to Polars dataframe
    if trade_history:
        trades_df = pl.DataFrame(trade_history)
    else:
        trades_df = pl.DataFrame()

In [ ]:
# Resample data to reduce points for plotting
# Target: max 10000 points or resample to 5-minute intervals, whichever gives fewer points
MAX_PLOT_POINTS = 10000
RESAMPLE_INTERVAL = "5m"  # 5 minutes

original_length = len(df_backtest)
print(f"Original data points: {original_length}")

# Calculate time range to determine appropriate resampling
if original_length > MAX_PLOT_POINTS:
    # Resample to reduce data points
    df_backtest_plot = df_backtest.sort("timestamp").with_columns(
        pl.col("timestamp").cast(pl.Datetime)
    )

    # Try resampling by time interval first
    try:
        df_backtest_plot = df_backtest_plot.group_by_dynamic(
            "timestamp", every=RESAMPLE_INTERVAL, closed="left", label="left"
        ).agg(
            [
                pl.first("timestamp"),
                pl.mean("spread"),
                pl.first("p_value"),  # Keep first p_value in interval
                pl.first("ou_theta"),
                pl.first("entry_level_1"),
                pl.first("exit_level_1"),
                pl.first("loss_level_1"),
                pl.first("entry_level_2"),
                pl.first("exit_level_2"),
                pl.first("loss_level_2"),
            ]
        )

        # If still too many points, downsample further
        if len(df_backtest_plot) > MAX_PLOT_POINTS:
            step = max(1, len(df_backtest_plot) // MAX_PLOT_POINTS)
            df_backtest_plot = df_backtest_plot[::step]

    except Exception as e:
        # Fallback: simple downsampling by taking every Nth row
        print(f"Time-based resampling failed: {e}, using simple downsampling")
        step = max(1, original_length // MAX_PLOT_POINTS)
        df_backtest_plot = df_backtest.sort("timestamp")[::step]

    print(
        f"Resampled data points: {len(df_backtest_plot)} ({100 * len(df_backtest_plot) / original_length:.1f}% of original)"
    )
    df_backtest = df_backtest_plot
else:
    print("Data points within limit, no resampling needed")

In [ ]:
# Calculate buy-and-hold comparison: hold both currencies equally
# Get first and last prices from df_backtest
first_row = df_backtest.slice(0, 1)
last_row = df_backtest.slice(len(df_backtest) - 1, 1)

first_price_1 = first_row["close_1"].item()
first_price_2 = first_row["close_2"].item()
last_price_1 = last_row["close_1"].item()
last_price_2 = last_row["close_2"].item()

# Buy equal amounts of both currencies at the start
bh_position_value = initial_cash * 0.5  # 50% in each currency
bh_shares_1 = bh_position_value / first_price_1
bh_shares_2 = bh_position_value / first_price_2

# Apply transaction costs for initial purchase
# When buying, you pay: price * (1 + TRANSACTION_COST) per share
cost_1 = bh_shares_1 * first_price_1 * (1 + TRANSACTION_COST)
cost_2 = bh_shares_2 * first_price_2 * (1 + TRANSACTION_COST)
bh_cash_after_purchase = initial_cash - cost_1 - cost_2

# Calculate final value (mark to market)
# Current value of holdings at end prices
bh_final_value_1 = bh_shares_1 * last_price_1
bh_final_value_2 = bh_shares_2 * last_price_2
# Final portfolio value = remaining cash + current value of holdings
bh_final_portfolio_value = bh_cash_after_purchase + bh_final_value_1 + bh_final_value_2

bh_total_return = bh_final_portfolio_value - initial_cash
bh_return_pct = (bh_total_return / initial_cash) * 100

print("\n===== Buy-and-Hold Comparison (Equal Weights) =====")
print(f"Initial Cash: ${initial_cash:,.2f}")
print("\nCurrency 1:")
print(f"  Starting Price: ${first_price_1:.4f}")
print(f"  Ending Price: ${last_price_1:.4f}")
print(
    f"  Price Change: ${last_price_1 - first_price_1:.4f} ({((last_price_1 / first_price_1 - 1) * 100):.2f}%)"
)
print(f"  Shares Purchased: {bh_shares_1:.6f}")
print("\nCurrency 2:")
print(f"  Starting Price: ${first_price_2:.4f}")
print(f"  Ending Price: ${last_price_2:.4f}")
print(
    f"  Price Change: ${last_price_2 - first_price_2:.4f} ({((last_price_2 / first_price_2 - 1) * 100):.2f}%)"
)
print(f"  Shares Purchased: {bh_shares_2:.6f}")
print(f"\nFinal Value (Mark-to-Market): ${bh_final_portfolio_value:,.2f}")
print(f"Total Return: ${bh_total_return:,.2f} ({bh_return_pct:.2f}%)")

# Calculate alpha (excess return over benchmark)
# Alpha = Strategy Return - Benchmark Return
alpha_dollar = total_return - bh_total_return
alpha_pct = return_pct - bh_return_pct

# Calculate portfolio returns over time for risk metrics
# Buy-and-hold: simple mark-to-market returns
bh_returns = []
for i in range(1, len(df_backtest)):
    prev_price_1 = df_backtest.slice(i - 1, 1)["close_1"].item()
    prev_price_2 = df_backtest.slice(i - 1, 1)["close_2"].item()
    curr_price_1 = df_backtest.slice(i, 1)["close_1"].item()
    curr_price_2 = df_backtest.slice(i, 1)["close_2"].item()

    prev_value = bh_shares_1 * prev_price_1 + bh_shares_2 * prev_price_2
    curr_value = bh_shares_1 * curr_price_1 + bh_shares_2 * curr_price_2
    if prev_value > 0:
        bh_returns.append((curr_value - prev_value) / prev_value)
    else:
        bh_returns.append(0)

# Pairs trading: track portfolio values over time to calculate returns
# Build a map of entry/exit indices for quick lookup
entry_map = {t["index"]: t for t in trade_history if t["type"] == "entry"}
exit_map = {t["index"]: t for t in trade_history if t["type"] == "exit"}

# Track portfolio values - calculate BEFORE and AFTER each trade
pairs_portfolio_values = []
pairs_cash_track = initial_cash
pairs_in_position_track = False
pairs_current_direction = None  # Track which direction the current position is
pairs_shares_short_track = 0
pairs_shares_long_track = 0
pairs_entry_price_short_track = 0
pairs_entry_price_long_track = 0

for i in range(len(df_backtest)):
    # Get current prices first (needed for portfolio value calculation)
    price_1 = df_backtest.slice(i, 1)["close_1"].item()
    price_2 = df_backtest.slice(i, 1)["close_2"].item()

    # Determine price_short/price_long based on current position direction
    # Direction 1: short close_2, long close_1
    # Direction 2: short close_1, long close_2
    if pairs_current_direction == 1:
        price_short, price_long = price_2, price_1
    elif pairs_current_direction == 2:
        price_short, price_long = price_1, price_2
    else:
        price_short, price_long = price_1, price_2  # Default (not in position)

    # Handle entry/exit at this index
    if i in entry_map:
        trade = entry_map[i]
        pairs_current_direction = trade["direction"]
        # Update prices based on the trade's direction
        if pairs_current_direction == 1:
            price_short, price_long = price_2, price_1
        else:
            price_short, price_long = price_1, price_2

        # When entering: we pay for the position + transaction costs
        # The position value at entry prices equals what we pay (minus transaction costs)
        # So portfolio value drops only by transaction costs
        position_cost = (
            trade["shares_short"] * trade["price_short"]
            + trade["shares_long"] * trade["price_long"]
        )
        transaction_costs = (
            trade["shares_short"] * trade["price_short"]
            + trade["shares_long"] * trade["price_long"]
        ) * TRANSACTION_COST
        pairs_cash_track -= position_cost + transaction_costs
        pairs_shares_short_track = trade["shares_short"]
        pairs_shares_long_track = trade["shares_long"]
        pairs_entry_price_short_track = trade["price_short"]
        pairs_entry_price_long_track = trade["price_long"]
        pairs_in_position_track = True

    elif i in exit_map:
        trade = exit_map[i]
        # Add P&L and subtract exit transaction costs
        pairs_cash_track += trade["total_pnl"]
        pairs_cash_track -= (
            pairs_shares_short_track * trade["price_short"] * TRANSACTION_COST
            + pairs_shares_long_track * trade["price_long"] * TRANSACTION_COST
        )
        pairs_shares_short_track = 0
        pairs_shares_long_track = 0
        pairs_in_position_track = False
        pairs_current_direction = None

    # Calculate portfolio value AFTER handling any trade at this index
    if pairs_in_position_track:
        # Portfolio value = cash + current position value
        # Current position value = (long position value) - (short position value)
        # = shares_long * current_price_long - shares_short * current_price_short
        # This is equivalent to: cash + (entry position value) + unrealized P&L
        # But we calculate it directly from current prices
        current_position_value = (
            pairs_shares_long_track * price_long
            - pairs_shares_short_track * price_short
        )
        portfolio_value = pairs_cash_track + current_position_value
    else:
        portfolio_value = pairs_cash_track

    pairs_portfolio_values.append(portfolio_value)

# Calculate returns from portfolio values
pairs_returns = []
for i in range(1, len(pairs_portfolio_values)):
    prev_value = pairs_portfolio_values[i - 1]
    curr_value = pairs_portfolio_values[i]
    if (
        prev_value > 0 and abs(prev_value) > 1e-10
    ):  # Avoid division by very small numbers
        ret = (curr_value - prev_value) / prev_value
        # Cap returns at reasonable levels to avoid outliers
        ret = max(-1.0, min(1.0, ret))  # Cap at -100% to +100%
        pairs_returns.append(ret)
    else:
        pairs_returns.append(0)

bh_returns = np.array(bh_returns)
pairs_returns = np.array(pairs_returns)

# Calculate volatility (annualized)
if len(df_backtest) > 1:
    first_ts = df_backtest.slice(0, 1)["timestamp"].item()
    last_ts = df_backtest.slice(len(df_backtest) - 1, 1)["timestamp"].item()
    days = (last_ts - first_ts).total_seconds() / (24 * 3600)
    periods_per_year = 252 / (days / len(df_backtest)) if days > 0 else 252
else:
    periods_per_year = 252

bh_vol = np.std(bh_returns) * np.sqrt(periods_per_year) if len(bh_returns) > 0 else 0
pairs_vol = (
    np.std(pairs_returns) * np.sqrt(periods_per_year) if len(pairs_returns) > 0 else 0
)

# Calculate Sharpe ratios (assuming risk-free rate = 0)
pairs_sharpe = (return_pct / 100) / pairs_vol if pairs_vol > 0 else 0
bh_sharpe = (bh_return_pct / 100) / bh_vol if bh_vol > 0 else 0

# Information Ratio (alpha / tracking error)
min_len = min(len(pairs_returns), len(bh_returns))
if min_len > 0:
    excess_returns = pairs_returns[:min_len] - bh_returns[:min_len]
    tracking_error = (
        np.std(excess_returns) * np.sqrt(periods_per_year)
        if len(excess_returns) > 0
        else 0
    )
    information_ratio = (alpha_pct / 100) / tracking_error if tracking_error > 0 else 0
else:
    tracking_error = 0
    information_ratio = 0

print("\n===== Strategy Comparison =====")
print(f"Pairs Trading Return: ${total_return:,.2f} ({return_pct:.2f}%)")
print(f"Buy-and-Hold Return: ${bh_total_return:,.2f} ({bh_return_pct:.2f}%)")
print(f"Excess Return: ${alpha_dollar:,.2f} ({alpha_pct:.2f}%)")
print("\n===== Risk Metrics =====")
print(f"Pairs Trading Volatility (annualized): {pairs_vol * 100:.2f}%")
print(f"Buy-and-Hold Volatility (annualized): {bh_vol * 100:.2f}%")
print(f"Pairs Trading Sharpe Ratio: {pairs_sharpe:.4f}")
print(f"Buy-and-Hold Sharpe Ratio: {bh_sharpe:.4f}")
print("\n===== Alpha Metrics =====")
print(f"Alpha (Excess Return): ${alpha_dollar:,.2f} ({alpha_pct:.2f}%)")
print(f"Tracking Error (annualized): {tracking_error * 100:.2f}%")
print(f"Information Ratio: {information_ratio:.4f}")

if total_return > bh_total_return:
    outperformance_pct = (
        ((return_pct - bh_return_pct) / abs(bh_return_pct) * 100)
        if bh_return_pct != 0
        else 0
    )
    print(f"\nPairs Trading outperformed Buy-and-Hold by {outperformance_pct:.2f}%")
else:
    outperformance_pct = (
        ((bh_return_pct - return_pct) / abs(return_pct) * 100) if return_pct != 0 else 0
    )
    print(f"\nBuy-and-Hold outperformed Pairs Trading by {outperformance_pct:.2f}%")

In [ ]:
# Check that we have the required data
if "df_backtest" not in locals():
    print("ERROR: df_backtest not found. Please run the previous cells first.")
elif "trades_df" not in locals() or len(trades_df) == 0:
    print("WARNING: No trades found. Plotting levels only.")
    trades_df = pl.DataFrame()

# Color scheme - different colors for each direction
COLORS = {
    # Shared
    "spread": "rgba(50, 50, 50, 0.9)",  # Dark gray for the shared spread
    "mean": "purple",
    # Direction 1 (blues/greens)
    "dir1_entry": "rgba(0, 128, 0, 0.8)",  # Green
    "dir1_exit": "rgba(0, 100, 200, 0.8)",  # Blue
    "dir1_loss": "rgba(0, 180, 180, 0.8)",  # Teal
    "dir1_trade_entry": "green",
    "dir1_trade_tp": "blue",
    "dir1_trade_sl": "teal",
    "dir1_connector": "rgba(0, 128, 0, 0.4)",
    # Direction 2 (reds/oranges)
    "dir2_entry": "rgba(200, 50, 50, 0.8)",  # Red
    "dir2_exit": "rgba(255, 140, 0, 0.8)",  # Orange
    "dir2_loss": "rgba(180, 0, 180, 0.8)",  # Magenta
    "dir2_trade_entry": "red",
    "dir2_trade_tp": "orange",
    "dir2_trade_sl": "magenta",
    "dir2_connector": "rgba(200, 50, 50, 0.4)",
}

# Get data from df_backtest (single spread and p_value, separate levels per direction)
timestamps = df_backtest["timestamp"].to_list()

# Single p_value and spread (used for both directions)
p_values = df_backtest["p_value"].to_list()
spread = df_backtest["spread"].to_list()
theta_values = (
    df_backtest["ou_theta"].to_list() if "ou_theta" in df_backtest.columns else None
)

# Direction 1 levels
entry_levels_1 = df_backtest["entry_level_1"].to_list()
exit_levels_1 = df_backtest["exit_level_1"].to_list()
loss_levels_1 = df_backtest["loss_level_1"].to_list()

# Direction 2 levels
entry_levels_2 = df_backtest["entry_level_2"].to_list()
exit_levels_2 = df_backtest["exit_level_2"].to_list()
loss_levels_2 = df_backtest["loss_level_2"].to_list()


# Helper function to mask invalid values based on single p_value
def mask_invalid_values(p_values, data_list):
    """Mask values where p_value is invalid."""
    result = list(data_list)
    for i, pv in enumerate(p_values):
        is_invalid = pv is None or (
            isinstance(pv, float) and (np.isnan(pv) or pv > P_VALUE_THRESHOLD)
        )
        if is_invalid:
            result[i] = None
    return result


# Mask invalid values (single p_value applies to all)
spread_masked = mask_invalid_values(p_values, spread)
theta_values_masked = (
    mask_invalid_values(p_values, theta_values) if theta_values is not None else None
)
entry_levels_1_masked = mask_invalid_values(p_values, entry_levels_1)
exit_levels_1_masked = mask_invalid_values(p_values, exit_levels_1)
loss_levels_1_masked = mask_invalid_values(p_values, loss_levels_1)
entry_levels_2_masked = mask_invalid_values(p_values, entry_levels_2)
exit_levels_2_masked = mask_invalid_values(p_values, exit_levels_2)
loss_levels_2_masked = mask_invalid_values(p_values, loss_levels_2)

# Count valid periods (single p_value for both directions)
valid_count = sum(
    1
    for pv in p_values
    if pv is not None
    and not (isinstance(pv, float) and (np.isnan(pv) or pv > P_VALUE_THRESHOLD))
)
print(
    f"Valid trading periods: {valid_count}/{len(p_values)} ({100 * valid_count / len(p_values):.1f}%)"
)


# Identify no-trading regions (where p_value is invalid)
def is_invalid(pv):
    return pv is None or (
        isinstance(pv, float) and (np.isnan(pv) or pv > P_VALUE_THRESHOLD)
    )


no_trading_regions = []
in_no_trading_region = False
region_start = None

for i in range(len(timestamps)):
    pv_invalid = is_invalid(p_values[i])

    if pv_invalid and not in_no_trading_region:
        region_start = timestamps[i]
        in_no_trading_region = True
    elif not pv_invalid and in_no_trading_region:
        no_trading_regions.append(
            (region_start, timestamps[i - 1] if i > 0 else timestamps[i])
        )
        in_no_trading_region = False
        region_start = None

# Close final region if still open
if in_no_trading_region and region_start is not None:
    no_trading_regions.append((region_start, timestamps[-1]))

print(f"Found {len(no_trading_regions)} no-trading period(s)")

# Create single figure (both directions on same plot)
fig = go.Figure()

# Add no-trading regions as shaded areas with vertical dashed borders
for idx, (start_ts, end_ts) in enumerate(no_trading_regions):
    # Add shaded region
    fig.add_vrect(
        x0=start_ts,
        x1=end_ts,
        fillcolor="gray",
        opacity=0.15,
        layer="below",
        line_width=0,
    )

    # Add vertical dashed lines at boundaries
    # Left boundary
    fig.add_vline(
        x=start_ts,
        line=dict(color="black", width=1, dash="dash"),
        opacity=0.5,
    )
    # Right boundary
    fig.add_vline(
        x=end_ts,
        line=dict(color="black", width=1, dash="dash"),
        opacity=0.5,
    )


# Plot shared spread (single line for both directions)
fig.add_trace(
    go.Scattergl(
        x=timestamps,
        y=spread_masked,
        mode="lines",
        name="Spread",
        line=dict(color=COLORS["spread"], width=1.5),
        opacity=0.9,
        connectgaps=False,
    )
)

# Plot mean (theta)
if theta_values_masked is not None:
    fig.add_trace(
        go.Scattergl(
            x=timestamps,
            y=theta_values_masked,
            mode="lines",
            name="Mean (θ)",
            line=dict(color=COLORS["mean"], width=1.5),
            opacity=0.7,
            connectgaps=False,
        )
    )

# Direction 1 levels (greens/blues - spread going DOWN)
fig.add_trace(
    go.Scattergl(
        x=timestamps,
        y=entry_levels_1_masked,
        mode="lines",
        name="Dir1 Entry (d')",
        line=dict(color=COLORS["dir1_entry"], width=1),
        connectgaps=False,
    )
)
fig.add_trace(
    go.Scattergl(
        x=timestamps,
        y=exit_levels_1_masked,
        mode="lines",
        name="Dir1 Exit (b')",
        line=dict(color=COLORS["dir1_exit"], width=1),
        connectgaps=False,
    )
)
fig.add_trace(
    go.Scattergl(
        x=timestamps,
        y=loss_levels_1_masked,
        mode="lines",
        name="Dir1 Loss (L)",
        line=dict(color=COLORS["dir1_loss"], width=1),
        connectgaps=False,
    )
)

# Direction 2 levels (reds/oranges - spread going UP)
fig.add_trace(
    go.Scattergl(
        x=timestamps,
        y=entry_levels_2_masked,
        mode="lines",
        name="Dir2 Entry (d')",
        line=dict(color=COLORS["dir2_entry"], width=1),
        connectgaps=False,
    )
)
fig.add_trace(
    go.Scattergl(
        x=timestamps,
        y=exit_levels_2_masked,
        mode="lines",
        name="Dir2 Exit (b')",
        line=dict(color=COLORS["dir2_exit"], width=1),
        connectgaps=False,
    )
)
fig.add_trace(
    go.Scattergl(
        x=timestamps,
        y=loss_levels_2_masked,
        mode="lines",
        name="Dir2 Loss (L)",
        line=dict(color=COLORS["dir2_loss"], width=1),
        connectgaps=False,
    )
)

# Add trades if available
if len(trades_df) > 0:
    # Direction 1 trades (greens/blues)
    dir1_trades = trades_df.filter(pl.col("direction") == 1)
    dir1_entries = dir1_trades.filter(pl.col("type") == "entry")
    dir1_exits = dir1_trades.filter(pl.col("type") == "exit")

    if len(dir1_entries) > 0:
        fig.add_trace(
            go.Scatter(
                x=dir1_entries["timestamp"].to_list(),
                y=dir1_entries["spread"].to_list(),
                mode="markers",
                name=f"Dir1 Entry ({len(dir1_entries)})",
                marker=dict(
                    color=COLORS["dir1_trade_entry"],
                    size=14,
                    symbol="triangle-down",
                    line=dict(color="darkgreen", width=2),
                ),
            )
        )

    if len(dir1_exits) > 0:
        dir1_exits_tp = dir1_exits.filter(pl.col("exit_reason") == "take_profit")
        dir1_exits_sl = dir1_exits.filter(pl.col("exit_reason") == "stop_loss")

        if len(dir1_exits_tp) > 0:
            fig.add_trace(
                go.Scatter(
                    x=dir1_exits_tp["timestamp"].to_list(),
                    y=dir1_exits_tp["spread"].to_list(),
                    mode="markers",
                    name=f"Dir1 TP ({len(dir1_exits_tp)})",
                    marker=dict(
                        color=COLORS["dir1_trade_tp"],
                        size=14,
                        symbol="circle",
                        line=dict(color="darkblue", width=2),
                    ),
                )
            )

        if len(dir1_exits_sl) > 0:
            fig.add_trace(
                go.Scatter(
                    x=dir1_exits_sl["timestamp"].to_list(),
                    y=dir1_exits_sl["spread"].to_list(),
                    mode="markers",
                    name=f"Dir1 SL ({len(dir1_exits_sl)})",
                    marker=dict(
                        color=COLORS["dir1_trade_sl"],
                        size=16,
                        symbol="x",
                        line=dict(width=3),
                    ),
                )
            )

        # Draw connector lines for direction 1
        if len(dir1_entries) > 0 and len(dir1_exits) > 0:
            entry_ts = dir1_entries["timestamp"].to_list()
            entry_sp = dir1_entries["spread"].to_list()
            exit_ts = dir1_exits["timestamp"].to_list()
            exit_sp = dir1_exits["spread"].to_list()
            for i in range(min(len(entry_ts), len(exit_ts))):
                fig.add_trace(
                    go.Scatter(
                        x=[entry_ts[i], exit_ts[i]],
                        y=[entry_sp[i], exit_sp[i]],
                        mode="lines",
                        line=dict(color=COLORS["dir1_connector"], width=2),
                        showlegend=False,
                        hoverinfo="skip",
                    )
                )

    # Direction 2 trades (reds/oranges)
    dir2_trades = trades_df.filter(pl.col("direction") == 2)
    dir2_entries = dir2_trades.filter(pl.col("type") == "entry")
    dir2_exits = dir2_trades.filter(pl.col("type") == "exit")

    if len(dir2_entries) > 0:
        fig.add_trace(
            go.Scatter(
                x=dir2_entries["timestamp"].to_list(),
                y=dir2_entries["spread"].to_list(),
                mode="markers",
                name=f"Dir2 Entry ({len(dir2_entries)})",
                marker=dict(
                    color=COLORS["dir2_trade_entry"],
                    size=14,
                    symbol="triangle-up",
                    line=dict(color="darkred", width=2),
                ),
            )
        )

    if len(dir2_exits) > 0:
        dir2_exits_tp = dir2_exits.filter(pl.col("exit_reason") == "take_profit")
        dir2_exits_sl = dir2_exits.filter(pl.col("exit_reason") == "stop_loss")

        if len(dir2_exits_tp) > 0:
            fig.add_trace(
                go.Scatter(
                    x=dir2_exits_tp["timestamp"].to_list(),
                    y=dir2_exits_tp["spread"].to_list(),
                    mode="markers",
                    name=f"Dir2 TP ({len(dir2_exits_tp)})",
                    marker=dict(
                        color=COLORS["dir2_trade_tp"],
                        size=14,
                        symbol="circle",
                        line=dict(color="darkorange", width=2),
                    ),
                )
            )

        if len(dir2_exits_sl) > 0:
            fig.add_trace(
                go.Scatter(
                    x=dir2_exits_sl["timestamp"].to_list(),
                    y=dir2_exits_sl["spread"].to_list(),
                    mode="markers",
                    name=f"Dir2 SL ({len(dir2_exits_sl)})",
                    marker=dict(
                        color=COLORS["dir2_trade_sl"],
                        size=16,
                        symbol="x",
                        line=dict(width=3),
                    ),
                )
            )

        # Draw connector lines for direction 2
        if len(dir2_entries) > 0 and len(dir2_exits) > 0:
            entry_ts = dir2_entries["timestamp"].to_list()
            entry_sp = dir2_entries["spread"].to_list()
            exit_ts = dir2_exits["timestamp"].to_list()
            exit_sp = dir2_exits["spread"].to_list()
            for i in range(min(len(entry_ts), len(exit_ts))):
                fig.add_trace(
                    go.Scatter(
                        x=[entry_ts[i], exit_ts[i]],
                        y=[entry_sp[i], exit_sp[i]],
                        mode="lines",
                        line=dict(color=COLORS["dir2_connector"], width=2),
                        showlegend=False,
                        hoverinfo="skip",
                    )
                )

# Update layout
fig.update_layout(
    title="Bidirectional Pairs Trading: Spread, Levels, and Trades<br><sup>Dir1 (green/blue): spread down | Dir2 (red/orange): spread up | Gray: no-trading periods</sup>",
    xaxis_title="Timestamp",
    yaxis_title="Spread Value",
    hovermode="x unified",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99,
        bgcolor="rgba(255, 255, 255, 0.9)",
    ),
    width=1600,
    height=800,
)

# Print trade summary and take profit / stop loss statistics
if len(trades_df) > 0:
    entries_1 = trades_df.filter(
        (pl.col("type") == "entry") & (pl.col("direction") == 1)
    )
    entries_2 = trades_df.filter(
        (pl.col("type") == "entry") & (pl.col("direction") == 2)
    )

    # All exits, direction 1 and 2
    exits_1 = trades_df.filter((pl.col("type") == "exit") & (pl.col("direction") == 1))
    exits_2 = trades_df.filter((pl.col("type") == "exit") & (pl.col("direction") == 2))

    # For each direction, split by exit_reason
    exits_1_sl = exits_1.filter(pl.col("exit_reason") == "stop_loss")
    exits_1_tp = exits_1.filter(pl.col("exit_reason") == "take_profit")
    exits_2_sl = exits_2.filter(pl.col("exit_reason") == "stop_loss")
    exits_2_tp = exits_2.filter(pl.col("exit_reason") == "take_profit")

    print(
        f"\nDirection 1 (spread down, short close_2, long close_1): {len(entries_1)} entries, {len(exits_1_tp)} take profits, {len(exits_1_sl)} stop losses"
    )
    print(
        f"Direction 2 (spread up, short close_1, long close_2): {len(entries_2)} entries, {len(exits_2_tp)} take profits, {len(exits_2_sl)} stop losses"
    )
else:
    print("No trades executed during this period")

fig.show()